<a href="https://colab.research.google.com/github/Jaswanth2075/AI_Lab/blob/NeuralNetworks/NationalityIdentificationUsingRNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!curl -O https://download.pytorch.org/tutorial/data.zip; unzip data.zip


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2814k  100 2814k    0     0  20.7M      0 --:--:-- --:--:-- --:--:-- 20.9M
Archive:  data.zip
   creating: data/
  inflating: data/eng-fra.txt        
   creating: data/names/
  inflating: data/names/Arabic.txt   
  inflating: data/names/Chinese.txt  
  inflating: data/names/Czech.txt    
  inflating: data/names/Dutch.txt    
  inflating: data/names/English.txt  
  inflating: data/names/French.txt   
  inflating: data/names/German.txt   
  inflating: data/names/Greek.txt    
  inflating: data/names/Irish.txt    
  inflating: data/names/Italian.txt  
  inflating: data/names/Japanese.txt  
  inflating: data/names/Korean.txt   
  inflating: data/names/Polish.txt   
  inflating: data/names/Portuguese.txt  
  inflating: data/names/Russian.txt  
  inflating: data/names/Scottish.txt  
  inflating: data/names/Spanish.txt  
  inflating

In [3]:
!pip install unidecode

In [4]:
import os
import random
from string import ascii_letters

import torch
from torch import nn
import torch.nn.functional as F
from unidecode import unidecode

_ = torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [22]:
data_dir = "./data/names"

lang2label = {
    file_name.split(".")[0]: torch.tensor(i, dtype=torch.long)
    for i, file_name in enumerate(os.listdir(data_dir))
}

num_langs = len(lang2label)

lang2label

{'Korean': tensor(0),
 'Scottish': tensor(1),
 'Portuguese': tensor(2),
 'French': tensor(3),
 'Czech': tensor(4),
 'German': tensor(5),
 'Greek': tensor(6),
 'Irish': tensor(7),
 'Japanese': tensor(8),
 'Polish': tensor(9),
 'Italian': tensor(10),
 'Russian': tensor(11),
 'Spanish': tensor(12),
 'English': tensor(13),
 'Dutch': tensor(14),
 'Chinese': tensor(15),
 'Vietnamese': tensor(16),
 'Arabic': tensor(17)}

In [13]:
char2idx = {letter: i for i, letter in enumerate(ascii_letters + " .'-")}
num_letters = len(char2idx)

In [9]:
def name2tensor(name):
  tensor = torch.zeros(len(name), 1, num_letters)

  for i, char in enumerate(name):
    tensor[i][0][char2idx[char]] = 1
  return tensor


In [14]:
name2tensor("abc")

tensor([[[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0.]],

        [[0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0.]],

        [[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0.]]])

Dataset Creation

In [15]:
tensor_names = []
tensor_langs = []
for file in os.listdir(data_dir):
  with open(os.path.join(data_dir, file)) as f:
    lang = file.split(".")[0]
    names = [unidecode(line.rstrip()) for line in f]
    for name in names:
      try:
        tensor_names.append(name2tensor(name))
        tensor_langs.append(lang2label[lang])
      except KeyError:
        pass

In [16]:
from sklearn.model_selection import train_test_split

train_idx, test_idx = train_test_split(
    range(len(tensor_langs)),
    test_size=0.1,
    shuffle = True,
    stratify = tensor_langs
)

train_dataset = [
    (tensor_names[i], tensor_langs[i])
    for i in train_idx
]

test_dataset = [
    (tensor_names[i], tensor_langs[i])
    for i in test_idx
]

In [17]:
print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

Train size: 18059
Test size: 2007


RNN


In [24]:
class MyRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MyRNN, self).__init__()
        self.hidden_size = hidden_size
        self.in2hidden = nn.Linear(input_size + hidden_size, hidden_size)
        self.in2output = nn.Linear(input_size + hidden_size, output_size)

    def forward(self, x, hidden_state):
        combined = torch.cat((x, hidden_state), 1)
        hidden = torch.sigmoid(self.in2hidden(combined))
        output = self.in2output(combined)
        return output, hidden

    def init_hidden(self):
        return nn.init.kaiming_uniform_(torch.empty(1, self.hidden_size))

In [25]:
hidden_size = 256
learning_rate = 0.001

model = MyRNN(num_letters, hidden_size, num_langs)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [26]:
hidden_size = 256
learning_rate = 0.001

model = MyRNN(num_letters, hidden_size, num_langs)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [29]:
num_epochs = 2
print_interval = 3000

for epoch in range(num_epochs):
    random.shuffle(train_dataset)
    for i, (name, label) in enumerate(train_dataset):
        hidden_state = model.init_hidden()
        for char in name:
            output, hidden_state = model(char, hidden_state)
        loss = criterion(output, label.unsqueeze(0))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()

        if (i + 1) % print_interval == 0:
            print(
                f"Epoch [{epoch + 1}/{num_epochs}], "
                f"Step [{i + 1}/{len(train_dataset)}], "
                f"Loss: {loss.item():.4f}"
            )

Epoch [1/2], Step [3000/18059], Loss: 1.3977
Epoch [1/2], Step [6000/18059], Loss: 1.9300
Epoch [1/2], Step [9000/18059], Loss: 0.2042
Epoch [1/2], Step [12000/18059], Loss: 3.1018
Epoch [1/2], Step [15000/18059], Loss: 0.0328
Epoch [1/2], Step [18000/18059], Loss: 0.5874
Epoch [2/2], Step [3000/18059], Loss: 3.2022
Epoch [2/2], Step [6000/18059], Loss: 4.0716
Epoch [2/2], Step [9000/18059], Loss: 0.2879
Epoch [2/2], Step [12000/18059], Loss: 2.0491
Epoch [2/2], Step [15000/18059], Loss: 0.1809
Epoch [2/2], Step [18000/18059], Loss: 0.2112


In [30]:
num_correct = 0
num_samples = len(test_dataset)

model.eval()

with torch.no_grad():
    for name, label in test_dataset:
        hidden_state = model.init_hidden()
        for char in name:
            output, hidden_state = model(char, hidden_state)
        _, pred = torch.max(output, dim=1)
        num_correct += bool(pred == label)

print(f"Accuracy: {num_correct / num_samples * 100:.4f}%")

Accuracy: 72.5959%


In [31]:
label2lang = {label.item(): lang for lang, label in lang2label.items()}

def myrnn_predict(name):
    model.eval()
    tensor_name = name2tensor(name)
    with torch.no_grad():
        hidden_state = model.init_hidden()
        for char in tensor_name:
            output, hidden_state = model(char, hidden_state)
        _, pred = torch.max(output, dim=1)
    model.train()
    return label2lang[pred.item()]

In [35]:
myrnn_predict("Sui")


'Chinese'